# Real-World MICE Imputation on MAR Data

In this notebook, we will use the real-world **California Housing Dataset**. We will simulate a scenario where the `HouseAge` data is missing for lower-income neighborhoods (a classic **Missing At Random (MAR)** scenario). We will then use **MICE (Multivariate Imputation by Chained Equations)** to intelligently reconstruct the missing data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression

# 1. Load the real-world California Housing Dataset
california = fetch_california_housing()
df = pd.DataFrame(california.data, columns=california.feature_names)

# For simplicity, let's just keep a few highly correlated columns
df = df[['MedInc', 'HouseAge', 'AveRooms', 'Population']]

print("Original Dataset Shape:", df.shape)
display(df.head())

### Simulating Missing At Random (MAR) Data
We will pretend that the `HouseAge` data was lost for 50% of the neighborhoods where the Median Income (`MedInc`) is very low. 
Because the missingness depends entirely on the `MedInc` column, this perfectly fits the definition of **MAR**.

In [ ]:
# 2. Introduce MAR Missingness
df_missing = df.copy()

# Find lower-income neighborhoods (e.g., bottom 30% of income)
low_income_threshold = df_missing['MedInc'].quantile(0.30)

# Randomly drop 50% of HouseAge records ONLY for those low-income neighborhoods
np.random.seed(42)
mask = (df_missing['MedInc'] <= low_income_threshold) & (np.random.rand(len(df_missing)) < 0.5)
df_missing.loc[mask, 'HouseAge'] = np.nan

print("Total missing values in HouseAge:", df_missing['HouseAge'].isna().sum())
display(df_missing[df_missing['HouseAge'].isna()].head())

### Applying MICE
Now we apply `IterativeImputer`. MICE will look at the relationships between `MedInc`, `AveRooms`, and `Population` to predict what the missing `HouseAge` values should logically be.

In [ ]:
# 3. Run MICE Imputation
mice_imputer = IterativeImputer(estimator=LinearRegression(), random_state=42, max_iter=10)

# Fit and transform the dataset
imputed_data = mice_imputer.fit_transform(df_missing)

# Convert back to DataFrame
df_imputed = pd.DataFrame(imputed_data, columns=df_missing.columns)

# We round the HouseAge because years are typically whole numbers
df_imputed['HouseAge'] = df_imputed['HouseAge'].round()

print("Missing values after MICE:", df_imputed['HouseAge'].isna().sum())
display(df_imputed.iloc[df_missing[df_missing['HouseAge'].isna()].index].head())

### Let's verify the accuracy!
We can now compare the Hidden True Values against the MICE Imputed Values.

In [ ]:
# 4. Compare True vs Imputed
missing_indices = df_missing[df_missing['HouseAge'].isna()].index

comparison = pd.DataFrame({
    'MedInc': df.loc[missing_indices, 'MedInc'],
    'True_HouseAge': df.loc[missing_indices, 'HouseAge'],
    'MICE_Imputed_Age': df_imputed.loc[missing_indices, 'HouseAge']
})

comparison['Error'] = abs(comparison['True_HouseAge'] - comparison['MICE_Imputed_Age'])

print(f"Average Error (in years): {comparison['Error'].mean():.2f}")
display(comparison.head(10))